## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### Service 패키지

#### 1. Server 

(py_srvcli/py_srvcli/service_member_function.py 참고.)

In [ ]:
from example_interfaces.srv import AddTwoInts
import rclpy
from rclpy.node import Node

class MinimalService(Node):
    def __init__(self):
        super().__init__('minimal_service')
        self.srv = self.create_service(AddTwoInts, 'add_two_ints', self.add_two_ints_callback)

    def add_two_ints_callback(self, request, response):
        response.sum = request.a + request.b
        self.get_logger().info('Incoming request\na: %d b: %d' % (request.a, request.b))
        return response
    
def main():
    rclpy.init()
    minimal_service = MinimalService()
    rclpy.spin(minimal_service)
    rclpy.shutdown()
    
if __name__ == '__main__':
    main()

Topic 패키지 코드와 동일한 부분은 생략한다.

In [ ]:
from example_interfaces.srv import AddTwoInts

example_interfaces 패키지 안 srv 폴더에서 AddTwoInts 파일을 사용하겠다는 뜻.

참고로 example_interfaces 역시 ROS2 기본 내장 패키지이며

AddTwoInts 파일을 열어보면 아래와 같은 내용으로 되어 있다.

(`cat /opt/ros/humble/share/example_interfaces/srv/AddTwoInts.srv` 로 직접 확인해도 된다.)

```bash
int64 a
int64 b
---
int64 sum
```


In [ ]:
import rclpy
from rclpy.node import Node

ROS2 기본 패키지 중 하나인 `rclpy`와 노드 기능들을 사용가능하게 하는 `Node`클래스를 사용하는 코드.

In [ ]:
class MinimalService(Node):

`Node` 클래스를 상속받는 Publisher 노드 클래스 `MinimalPublisher`를 만든다.

In [ ]:
self.srv = self.create_service(AddTwoInts, 'add_two_ints', self.add_two_ints_callback)

ROS2 Service Server를 `self.srv`로 생성하는 코드.

`AddTwoInts` : 서비스 타입. 구조는 위에 나와있다.

`'add_two_ints'` : 서비스 이름. 클라이언트는 이 이름으로 요청해야 한다.

가령 클라이언트는 아래와 같은 방식의 명령어로 요청을 하게 되는데 여기서 사용되는 서비스 이름이 된다.

```bash
ros2 service call /add_two_ints example_interfaces/srv/AddTwoInts "{a: 3, b: 5}"
```
topic 통신의 토픽 이름 같은 것으로 생각하면 편하다.

`self.add_two_ints_callback` : 요청이 들어왔을 때 실행될 콜백 함수.

add_two_ints_callback 함수

In [ ]:
def add_two_ints_callback(self, request, response):

srv 파일은 \-\-\- 기준으로 request와 response로 구성되어 있는데

srv 파일 안 변수들을 각각 request와 response의 속성으로 불러내기 위해

매개변수 request와 response를 사용한다.

In [ ]:
response.sum = request.a + request.b

request의 변수 a와 b의 합을 response의 sum 변수에 저장한다.

In [ ]:
self.get_logger().info('Incoming request\na: %d b: %d' % (request.a, request.b))

클라이언트로부터 온 값 a, b를 터미널에 출력하는 코드.

In [ ]:
return response

response를 반환함으로써 `self.srv`가 이를 다시 클라이언트에게 전송한다.

main함수는 topic 패키지와 동일한 구조이므로 생략한다.

#### 2. Client

(py_srvcli/py_srvcli/client_member_function.py 참고.)

In [ ]:
import sys
from example_interfaces.srv import AddTwoInts
import rclpy
from rclpy.node import Node

class MinimalClientAsync(Node):
    def __init__(self):
        super().__init__('minimal_client_async')
        self.cli = self.create_client(AddTwoInts, 'add_two_ints')
        while not self.cli.wait_for_service(timeout_sec=1.0):
            self.get_logger().info('service not available, waiting again...')
        self.req = AddTwoInts.Request()

    def send_request(self, a, b):
        self.req.a = a
        self.req.b = b
        return self.cli.call_async(self.req)
    
def main():
    rclpy.init()
    minimal_client = MinimalClientAsync()
    future = minimal_client.send_request(int(sys.argv[1]), int(sys.argv[2]))
    rclpy.spin_until_future_complete(minimal_client, future)
    response = future.result()
    minimal_client.get_logger().info(
        'Result of add_two_ints: for %d + %d = %d' %
        (int(sys.argv[1]), int(sys.argv[2]), response.sum))
    minimal_client.destroy_node()
    rclpy.shutdown()
    
if __name__ == '__main__':
    main()

Topic 패키지 코드 또는 Server 코드와 동일한 부분은 생략한다.

In [ ]:
import sys

ros2 명령어를 아래처럼 실행할 때

```bash
ros2 run my_pkg client 3 5
```

뒤에 붙은 인자들(3, 5)를 받기 위한 모듈 사용이다.

실제로 코드에서 sys.argv[1], sys.argv[2]로 받고 있다.

In [ ]:
self.cli = self.create_client(AddTwoInts, 'add_two_ints')

`add_two_ints`라는 Service 요청을 서버에게 보낼 수 있는 Client 객체를 만드는 코드.

사용할 서비스 타입은 당연히 `AddTwoInts`

In [ ]:
while not self.cli.wait_for_service(timeout_sec=1.0):
    self.get_logger().info('service not available, waiting again...')

`self.cli.wait_for_service(timeout_sec=1.0)` : `/add_two_ints`라는 Service 서버가 존재하는지 1초 동안 기다려본다.

리턴값은 존재하면 True, 아니면 False이다.

즉 이 코드는 `/add_two_ints` Service 서버가 존재하는지 1초마다 확인하고, 없으면 무한으로 대기 안내문구를 띄우는 코드이다.

`timeout_sec` 매개변수는 첫 번째 매개변수로 그냥 1.0만 () 안에 넣어도 문제없다.

In [ ]:
self.req = AddTwoInts.Request()

요청(request) 객체를 생성한다.

즉 내부적으로 request.a, request.b를 담은 객체를 생성하는 것이다.

In [ ]:
def send_request(self, a, b):
    self.req.a = a
    self.req.b = b
    return self.cli.call_async(self.req)

서비스 요청을 보내는 함수를 정의한다.

`call_async(self.req)` : Service 서버에게 요청을 비동기 방식으로 보내는 함수.

참고로 서버에게 요청을 동기 방식으로 보내는 `call()`이라는 함수가 있는데

`call()`과 `call_async()`과의 가장 큰 차이점은

비동기 방식 `call_async()`는 요청을 보내고 바로 다음 코드를 실행하며, 응답은 나중에 받지만,

동기 방식 `call()`는 요청을 보내고 응답이 올 때까지 멈춰서 기다리는 방식이다.

이런 이유로 일반적으로 서비스 통신에서 비동기 방식 `call_async()`가 더 많이 사용된다.

main함수

In [ ]:
future = minimal_client.send_request(int(sys.argv[1]), int(sys.argv[2]))

linux 명령어의 인자를 정수로 변환해서 서비스 요청을 전송한다.

비동기 방식은 요청을 보내고 응답이 오기 전 다음 코드를 먼저 수행하는 방식이기 때문에

반환값 객체 `future`는 나중에 응답이 들어올 예정인 결과 보관함이 된다.

In [ ]:
rclpy.spin_until_future_complete(minimal_client, future)

응답이 올 때까지 minimal_client를 실행하는 코드이며,

응답이 오면 이를 `future`객체에 저장한다.

`rclpy.spin()`과 `rclpy.spin_until_future_complete()`의 차이점은

`rclpy.spin()`은 노드 무한 실행이라 사용자가 직접 종료해야하지만

`rclpy.spin_until_future_complete()`은 응답이 올 때까지만 무한 실행이라 응답이 도착하면 자동으로 종료된다.

In [ ]:
response = future.result()

Future 객체는 나중에 도착할 결과(response)를 보관하는 객체이다.

실제 응답값(sum)은 Future 내부에 저장된 response 객체 안에 존재한다.

따라서 future.sum은 불가능하고, future.result()로 실제 response 객체를 꺼낸 뒤 response.sum 형태로 접근해야 한다.

#### 3. setup.py 내용추가

setup.py의 entry_points 안의 'console_scripts'에 아래와 같이 내용을 추가하여 ros2 명령어를 사용할 수 있게 만든다.

In [ ]:
entry_points={
    'console_scripts': [
        'service = py_srvcli.service_member_function:main',    # 추가내용
        'client = py_srvcli.client_member_function:main',      # 추가내용
    ],
}

package.xml에도 의존하는 패키지들을 추가한다.

(topic_package.ipynb에 나온 내용 참고.)

#### 4. 빌드

```bash
cd ~/ros2_ws
```

워크스페이스로 돌아와서

```bash
colcon build
```

빌드를 해준다.(--symlink-install 옵션 넣어도 된다.(권장))

#### 5. 실행

터미널 2개를 띄워서

양쪽 모두 아래 명령어를 실행하고

```bash
source install/setup.bash
```

한쪽에는 

```bash
ros2 run py_srvcli service
```

반대쪽에는 

```bash
ros2 run py_srvcli client 2 3
```

를 실행한다.

인자가 잘 전송이 되었는지, 합이 잘 들어왔는지 확인한다.

Action Package 실습코드

[Action_package](../ros2기초/action_package.ipynb)